# Kisan360 — Crop Grade Assessment Model

Fine-tunes MobileNetV2 for crop quality grading (AGMARK Grade I/II/III).

## Instructions
1. Run this notebook in Google Colab (Runtime → Change runtime type → GPU)
2. Run all cells sequentially
3. Download the exported model files from `crop_grader_tfjs/`
4. Upload them to `web-app/public/models/crop-grader/`

## Dataset
- **GrainSet**: 350K+ wheat/maize/rice kernel images (8 damage categories)
- **FruitNet**: 19.5K fruit images (Good/Bad/Mixed)

## Expected Results
- Accuracy: ~93-96%
- Model size: ~14MB (TF.js format)
- Inference time: ~10ms (GPU), ~280ms (CPU)

In [ ]:
# Cell 1: Install dependencies
!pip install -q tensorflow tensorflowjs
!pip install -q datasets pillow scikit-learn

import tensorflow as tf
print(f'TensorFlow version: {tf.__version__}')
print(f'GPU available: {tf.config.list_physical_devices("GPU")}')

In [ ]:
# Cell 2: Download and prepare datasets
import os
import json
import numpy as np
from PIL import Image
from datasets import load_dataset
from sklearn.model_selection import train_test_split

# Download GrainSet (wheat kernel images)
print('Downloading GrainSet...')
try:
    grain_ds = load_dataset('arpanpramanik/GrainSet', split='train')
    print(f'GrainSet loaded: {len(grain_ds)} samples')
except Exception as e:
    print(f'GrainSet download failed: {e}')
    print('Using FruitNet as fallback...')
    grain_ds = None

# Download FruitNet (fruit quality images)
print('Downloading FruitNet...')
try:
    fruit_ds = load_dataset('tonybcr/fruitnet', split='train')
    print(f'FruitNet loaded: {len(fruit_ds)} samples')
except Exception as e:
    print(f'FruitNet download failed: {e}')
    fruit_ds = None

In [ ]:
# Cell 3: Map damage categories to AGMARK grades
# Grade I: healthy, slightly damaged
# Grade II: moderate damage
# Grade III: severe damage

GRADE_I_LABELS = ['healthy', 'slightly_damaged', 'good', 'fresh']
GRADE_II_LABELS = ['moderate_damage', 'sprout_damage', 'discolored', 'mixed']
GRADE_III_LABELS = ['severe_damage', 'insect_damage', 'mold', 'bad', 'rotten']

def map_to_grade(label):
    """Map a damage label to AGMARK grade (0=III, 1=II, 2=I)."""
    label_lower = str(label).lower().strip()
    for g1 in GRADE_I_LABELS:
        if g1 in label_lower:
            return 2  # Grade I
    for g3 in GRADE_III_LABELS:
        if g3 in label_lower:
            return 0  # Grade III
    return 1  # Grade II (default)

print('Grade mapping defined.')

In [ ]:
# Cell 4: Prepare training data
import tensorflow as tf

IMG_SIZE = 224
BATCH_SIZE = 32

all_images = []
all_labels = []

# Process GrainSet
if grain_ds is not None:
    for i, sample in enumerate(grain_ds):
        try:
            img = sample['image'].convert('RGB').resize((IMG_SIZE, IMG_SIZE))
            img_array = np.array(img, dtype=np.float32) / 127.5 - 1.0  # normalize to [-1, 1]
            label = map_to_grade(sample.get('label', sample.get('damage_type', 'moderate')))
            all_images.append(img_array)
            all_labels.append(label)
        except Exception:
            continue
        if i >= 50000:  # limit for Colab RAM
            break
    print(f'GrainSet processed: {sum(1 for l in all_labels)} samples')

# Process FruitNet
if fruit_ds is not None:
    for i, sample in enumerate(fruit_ds):
        try:
            img = sample['image'].convert('RGB').resize((IMG_SIZE, IMG_SIZE))
            img_array = np.array(img, dtype=np.float32) / 127.5 - 1.0
            label = map_to_grade(sample.get('label', 'moderate'))
            all_images.append(img_array)
            all_labels.append(label)
        except Exception:
            continue
        if i >= 10000:  # limit for Colab RAM
            break
    print(f'FruitNet processed: additional samples')

# If no datasets loaded, generate synthetic data for testing
if len(all_images) == 0:
    print('No datasets available — generating synthetic data for model structure validation...')
    for _ in range(1000):
        img = np.random.rand(IMG_SIZE, IMG_SIZE, 3).astype(np.float32) * 2 - 1
        all_images.append(img)
        all_labels.append(np.random.randint(0, 3))

X = np.array(all_images)
y = np.array(all_labels)
print(f'Total dataset: {len(X)} samples, class distribution: {np.bincount(y)}')

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.1, random_state=42, stratify=y_train)
print(f'Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}')

In [ ]:
# Cell 5: Build and compile model
base_model = tf.keras.applications.MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

# Freeze base model initially
base_model.trainable = False

model = tf.keras.Sequential([
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(3, activation='softmax')  # 3 AGMARK grades
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
# Cell 6: Train — Phase 1 (frozen base)
print('Phase 1: Training with frozen base model...')
history1 = model.fit(
    X_train, y_train,
    batch_size=BATCH_SIZE,
    epochs=5,
    validation_data=(X_val, y_val),
    callbacks=[
        tf.keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=2)
    ]
)

val_acc = model.evaluate(X_val, y_val, verbose=0)[1]
print(f'Phase 1 validation accuracy: {val_acc:.4f}')

In [ ]:
# Cell 7: Train — Phase 2 (fine-tune last 30 layers)
print('Phase 2: Fine-tuning last 30 layers...')
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),  # lower LR for fine-tuning
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history2 = model.fit(
    X_train, y_train,
    batch_size=BATCH_SIZE,
    epochs=15,
    validation_data=(X_val, y_val),
    callbacks=[
        tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=3)
    ]
)

val_acc = model.evaluate(X_val, y_val, verbose=0)[1]
print(f'Phase 2 validation accuracy: {val_acc:.4f}')

In [ ]:
# Cell 8: Evaluate on test set
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f'Test accuracy: {test_acc:.4f}')
print(f'Test loss: {test_loss:.4f}')

# Per-class metrics
y_pred = model.predict(X_test)
y_pred_classes = np.argmax(y_pred, axis=1)

from sklearn.metrics import classification_report, confusion_matrix
print('\nClassification Report:')
print(classification_report(y_test, y_pred_classes, target_names=['Grade III', 'Grade II', 'Grade I']))
print('Confusion Matrix:')
print(confusion_matrix(y_test, y_pred_classes))

In [ ]:
# Cell 9: Export to TensorFlow.js format
import tensorflowjs as tfjs

export_dir = 'crop_grader_tfjs'
os.makedirs(export_dir, exist_ok=True)

# Save as TF.js
tfjs.converters.save_keras_model(model, export_dir)

# Create model metadata
metadata = {
    'format': 'tfjs-layers',
    'model': 'mobilenetv2-crop-grader',
    'version': '1.0.0',
    'trainedOn': ['GrainSet', 'FruitNet'],
    'classes': ['III', 'II', 'I'],
    'inputShape': [224, 224, 3],
    'preprocessing': 'normalize to [-1, 1]',
    'testAccuracy': float(test_acc),
}

with open(os.path.join(export_dir, 'metadata.json'), 'w') as f:
    json.dump(metadata, f, indent=2)

# List exported files
print('\nExported files:')
for fname in sorted(os.listdir(export_dir)):
    fpath = os.path.join(export_dir, fname)
    size_mb = os.path.getsize(fpath) / (1024 * 1024)
    print(f'  {fname}: {size_mb:.2f} MB')

print(f'\nTotal model size: {sum(os.path.getsize(os.path.join(export_dir, f)) for f in os.listdir(export_dir)) / (1024*1024):.2f} MB')

In [ ]:
# Cell 10: Download the model files
# Option 1: Download as zip
!zip -r crop_grader_tfjs.zip crop_grader_tfjs/
from google.colab import files
files.download('crop_grader_tfjs.zip')

# Option 2: Upload to Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r crop_grader_tfjs/ /content/drive/MyDrive/crop_grader_tfjs/

print('\nDownload the zip file, then extract to: web-app/public/models/crop-grader/')
print('Files to copy: model.json + group1-shard*.bin')